In [1]:
from ndtools import staged_max_flow as smf

from pathlib import Path
import json

from rsr import rsr
import torch

# Multi-state reliability analysis of pressure regularisation plant

This notebook repeats the rule-finding process of `s01_find_rules.ipynb`, but each component now has three states (0, 1, 2) with probabilities (0.03, 0.05, 0.92) and remaining-capacity ratios (0.0, 0.5, 1.0).

## Load data

In [2]:
DATASET = Path(r"data")

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs_multi.json").read_text(encoding="utf-8"))

In [3]:
def s_fun(comps_st):
    flow, sys_st_str, min_comp_state = smf.sys_fun( comps_st, nodes, edges, probs_dict, target_flow = 90.0 )

    sys_st = 1 if sys_st_str == 's' else 0
    return flow, sys_st, min_comp_state

row_names = list(probs_dict.keys())
n_state = 3  # three states: 0, 1, 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
probs = [[probs_dict[n][str(s)]['p'] for s in range(n_state)] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)

## Get rules for system event

In [4]:
sys_upper_st = 1

result = rsr.run_ref_extraction_by_mcs(
    # Problem-specific callables / data
    sfun=s_fun,
    probs=probs,
    row_names=row_names,
    n_state=n_state,
    sys_upper_st=sys_upper_st,
    unk_prob_thres = 1e-5,
    output_dir="rsr_res_multi"
)

---
Round: 1, Unk. prob.: 1.000e+00
Surv probs: 0.000e+00, Fail probs: 0.000e+00
No. of non-dominant rules: 0, Survival rules: 0, Failure rules: 0
Failure sample found from sampling.
No. of existing rules removed:  0
New rule added. System state: 0, System value: 55.0. Total samples: 100000.
New rule (No. of conditions: 2): {'x10': ('<=', 1), 'x40': ('<=', 0), 'sys': ('<=', 0)}
Updated sys_vals: [55.0]
---
Round: 2, Unk. prob.: 1.000e+00
Surv probs: 0.000e+00, Fail probs: 0.000e+00
No. of non-dominant rules: 1, Survival rules: 0, Failure rules: 1
Survival sample found from sampling.
No. of existing rules removed:  0
New rule added. System state: 1, System value: 90.0. Total samples: 100000.
New rule (No. of conditions: 31): {'x2': ('>=', 1), 'x3': ('>=', 1), 'x5': ('>=', 1), 'x6': ('>=', 1), 'x8': ('>=', 1), 'x9': ('>=', 1), 'x10': ('>=', 1), 'x11': ('>=', 1), 'x12': ('>=', 1), 'x13': ('>=', 2), 'x15': ('>=', 1), 'x16': ('>=', 1), 'x17': ('>=', 1), 'x18': ('>=', 1), 'x19': ('>=', 1), '